# Successful vs Unsuccessful Grant Comparison

Run the scoring pipeline on all PDFs in `data/successful/` and `data/unsuccessful/`, print results, and export to Excel.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from qwen3_ollama import _Scorer, score_application
from src.all_type_parser.all_type_parser import parse_and_save
from src.pool.build_pool import build_chunk_pool
from src.scoring.pipeline import (
    OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE,
    SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE,
    _aggregate_overall,
    _aggregate_section,
    _build_scored_section,
    _generate_json_with_parse_retry,
    _normalize_model_section_output,
    load_rubric,
)

CRITERIA_PATH = PROJECT_ROOT / 'criteria_points.json'
SUCCESSFUL_DIR = PROJECT_ROOT / 'data' / 'successful'
UNSUCCESSFUL_DIR = PROJECT_ROOT / 'data' / 'unsuccessful'
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'results' / 'compare'
PARSED_CACHE_DIR = RESULTS_DIR / 'parsed_cache'

EXPERIMENT_OLLAMA_MODEL = 'qwen3.5:27b'
BASELINE_MAX_TOKENS = int(os.environ.get('WHOLE_JSON_BASELINE_MAX_TOKENS', '32768'))

SECTION_KEYS = [
    'general',
    'proposed_research',
    'training_development',
    'sites_support',
    'wpcc',
    'application_form',
]

for path in [RESULTS_DIR, PARSED_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root      :', PROJECT_ROOT)
print('Results dir       :', RESULTS_DIR)
print('Model             :', EXPERIMENT_OLLAMA_MODEL)
print('Baseline max tok  :', BASELINE_MAX_TOKENS)

In [ ]:
def parse_pdf_cached(pdf_path: Path, *, reparse: bool = False) -> tuple[dict, Path]:
    json_path = PARSED_CACHE_DIR / f'{pdf_path.stem}.json'
    if reparse or not json_path.exists():
        parse_and_save(str(pdf_path), str(json_path))
    parsed = json.loads(json_path.read_text(encoding='utf-8'))
    return parsed, json_path


def score_pdf(pdf_path: Path, *, scorer: _Scorer, reparse: bool = False) -> dict:
    parsed, parsed_json_path = parse_pdf_cached(pdf_path, reparse=reparse)
    artifact_dir = RESULTS_DIR / 'artifacts' / pdf_path.stem
    artifact_dir.mkdir(parents=True, exist_ok=True)
    result = score_application(
        parsed, CRITERIA_PATH,
        doc_id=pdf_path.stem, scorer=scorer, artifacts_dir=artifact_dir,
    )
    result['source_pdf'] = str(pdf_path)
    result['parsed_json'] = str(parsed_json_path)
    return result


def score_pdf_baseline(pdf_path: Path, *, scorer: _Scorer) -> dict:
    parsed, _ = parse_pdf_cached(pdf_path)
    doc_type = (parsed.get('doc_type') or '').lower()
    excluded_sections = OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE.get(doc_type, set())
    excluded_sub_ids = SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE.get(doc_type, set())

    rubric_sections = load_rubric(CRITERIA_PATH)
    pool_data = build_chunk_pool(parsed)
    pool_lookup = pool_data['pool_lookup']
    all_chunk_ids = list(pool_lookup)
    chunk_order = {cid: i for i, cid in enumerate(all_chunk_ids)}

    # Build one-shot prompt
    chunk_index = [
        {'chunk_id': cid, 'parser_section': m.get('parser_section', ''),
         'preview': ' '.join((m.get('text', '') or '').split())[:900]}
        for cid, m in pool_lookup.items()
    ]
    signal_props = lambda sigs: {
        s['sid']: {'type': 'integer', 'enum': [0, 1, 2, 3, 4, 5]} for s in sigs
    }
    chunk_item = {'type': 'string', 'enum': all_chunk_ids} if all_chunk_ids else {'type': 'string'}
    sec_props = {}
    for sec in rubric_sections:
        sub_props = {
            sub['sub_id']: {
                'type': 'object',
                'properties': {
                    'signals': {'type': 'object', 'properties': signal_props(sub['signals']),
                                'required': list(signal_props(sub['signals'])), 'additionalProperties': False},
                    'used_chunk_ids': {'type': 'array', 'items': chunk_item, 'maxItems': 5},
                    'pros': {'type': 'string'}, 'drawbacks': {'type': 'string'},
                },
                'required': ['signals', 'used_chunk_ids', 'pros', 'drawbacks'],
                'additionalProperties': False,
            }
            for sub in sec['sub_criteria']
        }
        sec_props[sec['section_key']] = {
            'type': 'object', 'properties': sub_props,
            'required': list(sub_props), 'additionalProperties': False,
        }
    schema = {'type': 'object', 'properties': sec_props,
              'required': list(sec_props), 'additionalProperties': False}

    messages = [
        {'role': 'system', 'content': (
            'You are a strict NIHR grant scoring baseline. '
            'Return JSON only and follow the schema exactly.'
        )},
        {'role': 'user', 'content': json.dumps({
            'task': 'one_shot_whole_parsed_json_baseline_scoring',
            'criteria': rubric_sections,
            'evidence_chunk_index': chunk_index,
            'parsed_application_json': parsed,
        }, ensure_ascii=False)},
    ]

    _, parsed_response, _ = _generate_json_with_parse_retry(
        scorer, messages, schema=schema, max_tokens=BASELINE_MAX_TOKENS, max_retries=0,
    )

    sections = []
    for rubric_section in rubric_sections:
        sk = rubric_section['section_key']
        raw = parsed_response.get(sk, {}) if isinstance(parsed_response, dict) else {}
        normalized = _normalize_model_section_output(raw, rubric_section, all_chunk_ids)
        sections.append(_build_scored_section(rubric_section, normalized, chunk_order,
                                               excluded_sub_ids=excluded_sub_ids))

    features = {s['section_key']: _aggregate_section(s, pool_lookup) for s in sections}
    section_weights = {s['section_key']: s['weight'] for s in sections}
    return {
        'overall': _aggregate_overall(features, section_weights, excluded_sections=excluded_sections),
        'features': features,
    }


def flatten_scores(result: dict, *, prefix: str) -> dict:
    overall = result.get('overall', {})
    row: dict = {f'{prefix}overall_score_100': round(float(overall.get('final_score_0to100', 0)), 2)}
    features = result.get('features', {})
    for section_key in SECTION_KEYS:
        sec = features.get(section_key, {}).get('overall', {})
        row[f'{prefix}{section_key}_score_100'] = round(float(sec.get('final_score_0to100', 0)), 2)
        row[f'{prefix}{section_key}_evidence_count'] = int(sec.get('evidence_count', 0))
    return row


print('Helper functions defined.')

In [ ]:
scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)

all_pdfs: list[tuple[Path, str]] = [
    *[(p, 'successful') for p in sorted(SUCCESSFUL_DIR.glob('*.pdf'))],
    *[(p, 'unsuccessful') for p in sorted(UNSUCCESSFUL_DIR.glob('*.pdf'))],
]

print(f'PDFs found: {len(all_pdfs)}')
print(f'  successful  : {sum(1 for _, c in all_pdfs if c == "successful")}')
print(f'  unsuccessful: {sum(1 for _, c in all_pdfs if c == "unsuccessful")}')

rows: list[dict] = []
failed: list[tuple[str, str]] = []

for idx, (pdf_path, category) in enumerate(all_pdfs, start=1):
    print(f'\n[{idx}/{len(all_pdfs)}] {pdf_path.name}  [{category}]')
    try:
        pipeline_result = score_pdf(pdf_path, scorer=scorer)
        baseline_result = score_pdf_baseline(pdf_path, scorer=scorer)

        row = {'pdf_name': pdf_path.name, 'category': category}
        row.update(flatten_scores(pipeline_result, prefix='pipeline_'))
        row.update(flatten_scores(baseline_result, prefix='baseline_'))
        rows.append(row)

        print(f'  pipeline overall : {row["pipeline_overall_score_100"]:.1f}/100')
        print(f'  baseline overall : {row["baseline_overall_score_100"]:.1f}/100')
        for section_key in SECTION_KEYS:
            p_score = row[f'pipeline_{section_key}_score_100']
            b_score = row[f'baseline_{section_key}_score_100']
            p_ev    = row[f'pipeline_{section_key}_evidence_count']
            b_ev    = row[f'baseline_{section_key}_evidence_count']
            print(f'    {section_key:<25} pipeline={p_score:5.1f} (ev={p_ev})  baseline={b_score:5.1f} (ev={b_ev})')
    except Exception as exc:
        print(f'  ERROR: {exc}')
        failed.append((pdf_path.name, str(exc)))

print(f'\n\nDone: {len(rows)}/{len(all_pdfs)} succeeded')
if failed:
    print(f'Failed ({len(failed)}):')
    for name, err in failed:
        print(f'  - {name}: {err}')

In [ ]:
df = pd.DataFrame(rows).sort_values(['category', 'pdf_name']).reset_index(drop=True)

# Interleave pipeline/baseline columns per section
section_cols = []
for k in SECTION_KEYS:
    section_cols += [
        f'pipeline_{k}_score_100', f'baseline_{k}_score_100',
        f'pipeline_{k}_evidence_count', f'baseline_{k}_evidence_count',
    ]
display_cols = [
    'pdf_name', 'category',
    'pipeline_overall_score_100', 'baseline_overall_score_100',
    *section_cols,
]

display(df[display_cols])

In [ ]:
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
excel_path = RESULTS_DIR / f'comparison_{timestamp}.xlsx'

df[display_cols].to_excel(excel_path, index=False, sheet_name='Results')

print(f'Exported to: {excel_path}')
print(f'  Rows: {len(df)}  |  Columns: {len(display_cols)}')